In [0]:
"""CRISPRbrain.org Ingestion Pipeline

Ingests CRISPR screen data from CRISPRbrain (Kampmann Lab, UCSF/Gladstone)
into Unity Catalog for use by the neuro-crispr-mcp app.

Sources:
  1. Cell 2022 supplementary tables (xlsx) — comprehensive screen results
  2. Direct scraping of CRISPRbrain Dash app for remaining screens

Target: <catalog>.<schema>.crisprbrain_screens
"""

from config.neuroplex_config import load_config
CFG = load_config()
CATALOG = CFG.catalog
SCHEMA = CFG.schema
TABLE = f"{CATALOG}.{SCHEMA}.crisprbrain_screens"

# Cell 2022 supplementary files (contain all screen results)
SUPP_URLS = [
    "https://ars.els-cdn.com/content/image/1-s2.0-S0092867422005979-mmc1.xlsx",
    "https://ars.els-cdn.com/content/image/1-s2.0-S0092867422005979-mmc2.xlsx",
    "https://ars.els-cdn.com/content/image/1-s2.0-S0092867422005979-mmc3.xlsx",
]

# Screen metadata extracted from CRISPRbrain.org
SCREEN_NAMES = [
    "Glutamatergic Neuron-CellRox-CRISPRi",
    "Glutamatergic Neuron-Day-14-Survival-CRISPRi",
    "Glutamatergic Neuron-Day-21-Survival-CRISPRi",
    "Glutamatergic Neuron-Day-28-Survival-CRISPRi",
    "Glutamatergic Neuron-FeRhoNox-CRISPRi",
    "Glutamatergic Neuron-Halo-TDP-43-CRISPRi",
    "Glutamatergic Neuron-Liperfluo-CRISPRi",
    "Glutamatergic Neuron-Lysotracker-CRISPRi",
    "Glutamatergic Neuron-No Antioxidants-Survival-CRISPRa",
    "Glutamatergic Neuron-No Antioxidants-Survival-CRISPRi",
    "Glutamatergic Neuron-PSAP KO-No Antioxidants-Survival-CRISPRa",
    "Glutamatergic Neuron-PSAP KO-No Antioxidants-Survival-CRISPRi",
    "Glutamatergic Neuron-PSAP KO-Survival-CRISPRa",
    "Glutamatergic Neuron-PSAP KO-Survival-CRISPRi",
    "Glutamatergic Neuron-Survival-CRISPRa",
    "Glutamatergic Neuron-Survival-CRISPRi",
    "Glutamatergic-Neuron-DAKO-CRISPRi-Retest",
    "Glutamatergic-Neuron-M204-CRISPRi-Retest",
    "Glutamatergic-Neuron-T22-CRISPRi-Retest",
    "Glutamatergic-Neuron-T22-CRISPRi",
    "Glutamatergic-Neuron-TOC1-CRISPRi-Retest",
    "Glutamatergic-Neuron-WT-DAKO-CRISPRi-Retest",
    "Glutamatergic-Neuron-WT-T22-CRISPRi-Retest",
    "HSPC-CD34-CRISPRn",
    "T Cell-CFSE-CRISPRn",
    "iAstrocyte-LAMP1-ITC-CRISPRi",
    "iAstrocyte-LAMP1-Veh-CRISPRi",
    "iAstrocyte-LysoTracker-ITC-CRISPRi",
    "iAstrocyte-LysoTracker-Veh-CRISPRi",
    "iAstrocyte-Phagocytosis-ITC-CRISPRi",
    "iAstrocyte-Phagocytosis-Veh-CRISPRi",
    "iAstrocyte-VCAM1-ITC-CRISPRi",
    "iPSC-Halo-TDP-43-CRISPRi",
    "iPSC-Survival-CRISPRi",
    "iTF-Microglia-Antigen-presenting-HLA-DMB-TF-library",
    "iTF-Microglia-Chemokine-CCL13-TF-library",
    "iTF-Microglia-Disease-associated-CD9-TF-library",
    "iTF-Microglia-Homeostatic-P2RY12-TF-library",
    "iTF-Microglia-Immune Activation-CRISPRi",
    "iTF-Microglia-Interferon-responsive-IFIT1-TF-library",
    "iTF-Microglia-Lipid-rich-BODIPY-TF-library",
    "iTF-Microglia-Phagocytosis-CRISPRa",
    "iTF-Microglia-Phagocytosis-CRISPRi",
    "iTF-Microglia-Survival-Proliferation-CRISPRi",
]

print(f"Target table: {TABLE}")
print(f"Known screens: {len(SCREEN_NAMES)}")

In [0]:
import requests
import pandas as pd
from io import BytesIO
import re

def parse_screen_name(name: str) -> dict:
    """Parse screen name into cell_type, phenotype, genotype, crispr_mode."""
    # Extract CRISPR mode (always last segment)
    crispr_mode = "unknown"
    for mode in ["CRISPRi", "CRISPRa", "CRISPRn", "inducible CRISPRi", "inducible CRISPRa"]:
        if mode in name:
            crispr_mode = mode
            break
    
    # Extract cell type (first segment)
    cell_type = "unknown"
    for ct in ["Glutamatergic Neuron", "Glutamatergic-Neuron", "iTF-Microglia", "iAstrocyte", "iPSC", "T Cell", "HSPC"]:
        if name.startswith(ct):
            cell_type = ct.replace("-", " ").replace("Glutamatergic Neuron", "Glutamatergic Neuron")
            break
    
    # Extract genotype
    genotype = "WT"
    for geno in ["PSAP KO", "MAPT-WT/V337M", "MAPT-WT/WT", "i11w-hT"]:
        if geno in name:
            genotype = geno
            break
    
    # Extract phenotype (middle segments)
    parts = name.replace(cell_type.replace(" ", "-"), "").replace(cell_type, "")
    parts = parts.replace(crispr_mode, "").replace(genotype, "")
    phenotype = parts.strip("-").strip(" ").strip("-")
    if not phenotype:
        phenotype = "Survival"  # default
    
    return {
        "cell_type": cell_type.strip(),
        "phenotype_name": phenotype.strip("-").strip(),
        "genotype": genotype,
        "crispr_mode": crispr_mode,
    }

# Download and parse supplementary files
all_dfs = []

for url in SUPP_URLS:
    fname = url.split("/")[-1]
    print(f"Downloading {fname}...")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    
    xls = pd.ExcelFile(BytesIO(r.content))
    print(f"  Sheets: {xls.sheet_names}")
    
    for sheet in xls.sheet_names:
        try:
            df = pd.read_excel(xls, sheet_name=sheet)
            # Check if it looks like screen data (has Gene column)
            if "Gene" in df.columns or "gene" in df.columns:
                # Normalize column names
                df.columns = [c.strip() for c in df.columns]
                if "gene" in df.columns:
                    df = df.rename(columns={"gene": "Gene"})
                
                df["source_file"] = fname
                df["sheet_name"] = sheet
                all_dfs.append(df)
                print(f"    ✅ {sheet}: {len(df)} rows, cols={list(df.columns)[:6]}")
            else:
                print(f"    ⏭ {sheet}: Not screen data (cols: {list(df.columns)[:4]})")
        except Exception as e:
            print(f"    ❌ {sheet}: {e}")

print(f"\nTotal dataframes collected: {len(all_dfs)}")
if all_dfs:
    print(f"Total rows: {sum(len(d) for d in all_dfs):,}")

In [0]:
# Download the directly available screen CSVs from GitHub
github_screens = {
    "Glutamatergic Neuron-Day-14-Survival-CRISPRi": 
        "https://raw.githubusercontent.com/cory-weller/CRISPRbrain-guides/main/data/Tian_et_al_2019_2.csv",
    "Glutamatergic Neuron-Survival-CRISPRi":
        "https://raw.githubusercontent.com/cory-weller/CRISPRbrain-guides/main/data/Tian_et_al_2020_15.csv",
}

github_dfs = []
for screen_name, url in github_screens.items():
    print(f"Downloading {screen_name}...")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    df = pd.read_csv(BytesIO(r.content))
    df["screen_name"] = screen_name
    df["source"] = "github_csv"
    github_dfs.append(df)
    print(f"  ✅ {len(df)} rows, columns: {list(df.columns)}")

print(f"\nGitHub screens: {len(github_dfs)} screens, {sum(len(d) for d in github_dfs):,} total rows")
if github_dfs:
    print(f"\nSample (Glutamatergic Neuron-Survival-CRISPRi):")
    print(github_dfs[-1].head())

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DoubleType
from pyspark.sql import functions as F
from datetime import datetime

# Build unified dataframe from GitHub CSVs (these have the cleanest schema)
# Schema: screen_name, gene, cell_type, genotype, crispr_mode, phenotype_name,
#         phenotype_score, pvalue, gene_score, hit_class, species, source

rows = []
for df in github_dfs:
    screen_name = df["screen_name"].iloc[0]
    meta = parse_screen_name(screen_name)
    
    for _, row in df.iterrows():
        rows.append({
            "screen_name": screen_name,
            "gene": str(row.get("Gene", "")).strip(),
            "cell_type": meta["cell_type"],
            "genotype": meta["genotype"],
            "crispr_mode": meta["crispr_mode"],
            "phenotype_name": meta["phenotype_name"],
            "phenotype_score": float(row.get("Phenotype", 0)) if pd.notna(row.get("Phenotype")) else None,
            "pvalue": float(row.get("P Value", 1)) if pd.notna(row.get("P Value")) else None,
            "gene_score": float(row.get("Gene Score", 0)) if pd.notna(row.get("Gene Score")) else None,
            "hit_class": str(row.get("Hit Class", "none")).strip() if pd.notna(row.get("Hit Class")) else "none",
            "species": "human",
            "source": "crisprbrain.org",
        })

# Also process supplementary data if it has screen-like format
for df in all_dfs:
    sheet = df["sheet_name"].iloc[0] if "sheet_name" in df.columns else "unknown"
    # Try to use sheet name as screen name or infer from content
    screen_name = sheet
    meta = parse_screen_name(screen_name) if any(c in screen_name for c in ["CRISPRi", "CRISPRa", "CRISPRn"]) else {
        "cell_type": "Glutamatergic Neuron", "phenotype_name": sheet, "genotype": "WT", "crispr_mode": "CRISPRi"
    }
    
    for _, row in df.iterrows():
        gene = str(row.get("Gene", row.get("gene", ""))).strip()
        if not gene or gene == "nan":
            continue
        rows.append({
            "screen_name": screen_name,
            "gene": gene,
            "cell_type": meta["cell_type"],
            "genotype": meta["genotype"],
            "crispr_mode": meta["crispr_mode"],
            "phenotype_name": meta["phenotype_name"],
            "phenotype_score": float(row.get("Phenotype", row.get("phenotype", row.get("score", 0)))) if pd.notna(row.get("Phenotype", row.get("phenotype", row.get("score")))) else None,
            "pvalue": float(row.get("P Value", row.get("p_value", row.get("pval", 1)))) if pd.notna(row.get("P Value", row.get("p_value", row.get("pval")))) else None,
            "gene_score": float(row.get("Gene Score", row.get("gene_score", 0))) if pd.notna(row.get("Gene Score", row.get("gene_score"))) else None,
            "hit_class": str(row.get("Hit Class", row.get("hit_class", "none"))).strip() if pd.notna(row.get("Hit Class", row.get("hit_class"))) else "none",
            "species": "human",
            "source": "crisprbrain.org",
        })

print(f"Total unified rows: {len(rows):,}")

# Convert to Spark DataFrame
pdf = pd.DataFrame(rows)
print(f"\nUnified columns: {list(pdf.columns)}")
print(f"Screens: {pdf['screen_name'].nunique()}")
print(f"Unique genes: {pdf['gene'].nunique()}")
print(f"\nScreen counts:")
print(pdf['screen_name'].value_counts())

# Create Spark DF
sdf = spark.createDataFrame(pdf)
sdf = sdf.withColumn("ingested_at", F.current_timestamp())
sdf.printSchema()

In [0]:
# Write to Delta table
sdf.write.mode("overwrite").saveAsTable(TABLE)

row_count = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE}").collect()[0]["n"]
screen_count = spark.sql(f"SELECT COUNT(DISTINCT screen_name) as n FROM {TABLE}").collect()[0]["n"]
gene_count = spark.sql(f"SELECT COUNT(DISTINCT gene) as n FROM {TABLE}").collect()[0]["n"]

print(f"✅ Table created: {TABLE}")
print(f"   Rows: {row_count:,}")
print(f"   Screens: {screen_count}")
print(f"   Genes: {gene_count:,}")

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {TABLE} IS 
    'CRISPRbrain.org CRISPR screen results. Human iPSC-derived neurons, microglia, astrocytes. '
    'Source: Kampmann Lab (UCSF/Gladstone). Cell types: Glutamatergic Neuron, iTF-Microglia, iAstrocyte, iPSC. '
    'Phenotypes: Survival, ROS, iron, lipids, lysosomes, tau aggregation, TDP-43, phagocytosis.'
""")

# Grant permissions to neuro-crispr-mcp SP
SP_ID = "7b7e2188-3c15-4302-bf7c-770ef2e54c69"
spark.sql(f"GRANT SELECT ON TABLE {TABLE} TO `{SP_ID}`")
print(f"\n✅ SELECT granted to SP {SP_ID}")

# Also grant to tahoe-100m-mcp SP for cross-referencing
TAHOE_SP = "1b52826b-7a90-488f-a605-c2468aaf3bf4"
spark.sql(f"GRANT SELECT ON TABLE {TABLE} TO `{TAHOE_SP}`")
print(f"✅ SELECT granted to Tahoe SP {TAHOE_SP}")

# Verify
print(f"\nSample data:")
spark.sql(f"SELECT * FROM {TABLE} WHERE hit_class != 'none' LIMIT 10").show(truncate=30)

In [0]:
%sql
-- Validate the crisprbrain_screens table
SELECT 
  screen_name,
  cell_type,
  crispr_mode,
  genotype,
  COUNT(*) as n_genes,
  SUM(CASE WHEN hit_class != 'none' THEN 1 ELSE 0 END) as n_hits,
  ROUND(AVG(phenotype_score), 3) as avg_phenotype
FROM dhbl_discovery_us_dev.genesis_schema.crisprbrain_screens
GROUP BY screen_name, cell_type, crispr_mode, genotype
ORDER BY n_genes DESC